In [1]:
import os, json
import pandas as pd
import openai
from dotenv import load_dotenv
from pathlib import Path
from tqdm import tqdm
import time

In [2]:
# Load API
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME     = "gpt-4o-mini-2024-07-18" 
TEMPERATURE    = 1.0
print("API Key loaded:", openai.api_key is not None)

# Load files
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_twostage_1shot.csv"
df = pd.read_csv(src_file)
# df = df.head(200)

API Key loaded: True


# Materials

In [4]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [5]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [6]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [7]:
CATEGORICAL_VALUES = {
    "Disaster Classification": DISASTER_CLASSIFICATION,
    "Magnitude": MAGNITUDE_INFORMATION
}

# Prompt

In [9]:
OUTPUT_FIELDS = {
  "Event": "fields name",
  "Geographical Information": "fields name",
  "Effect": "fields name"
}

In [10]:
OUTPUT_VALUES = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [11]:
DESCRIPTION_EXAMPLE = """
    The ongoing flood crisis in Bangladesh began in mid-May 2025, primarily triggered by heavy monsoon rainfall and upstream water flow from India. As of 1 June 2025, several districts, including Sylhet, Sunamganj, Moulvibazar, Habiganj, Netrokona, Noakhali, Bhola, Khagrachari, Bandarban and Rangamati have been severely affected. (Bangladesh Red Crescent Society, 2 Jun 2025)
"""

In [12]:
EXAMPLE_FIELDS = {
    "Event": ["Disaster Group", "Disaster Type", "Disaster Date (YMD)", "Cause"], 
    "Geographical Information": ["Country", "Geographical Level 1"], 
    "Effect": []
    }

In [13]:
EXAMPLE_VALUES = {
    "Event": {
        "Disaster Group": "Hydrological", 
        "Disaster Type": "Flood", 
        "Disaster Date (YMD)": "2025-05", 
        "Cause": "Heavy monsoon rainfall and upstream water flow from India"
    },
    "Geographical Information": {
        "Country": "Bangladesh", 
        "Geographical Level 1": ["Sylhet", "Sunamganj", "Moulvibazar", "Habiganj", "Netrokona", "Noakhali", "Bhola", "Khagrachari", "Bandarban", "Rangamati"]
    }, 
    "Effect": {}
}

In [14]:
EXAMPLE_STAGE1 = {
    "description": DESCRIPTION_EXAMPLE,
    "output_fields": EXAMPLE_FIELDS,
}

In [15]:
EXAMPLE_STAGE2 = {
    "description": DESCRIPTION_EXAMPLE,
    "output_values": EXAMPLE_VALUES
}

In [16]:
fields_extraction_prompt = f"""
        Please refer FIELD_SCHEMA first, then list every field that is
        mentioned (even once) or is marked as 'Required' in DISASTER_DESCRIPTION. 
        Output must contain field names only.
        Finally return a single JSON object that conforms precisely to OUTPUT_FIELDS.
        Below there is a reference EXAMPLE_STAGE1 for one disaster's description and its output fields.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        OUTPUT_FIELDS:
        {json.dumps(OUTPUT_FIELDS, ensure_ascii=False)}

        EXAMPLE_STAGE1:
        {json.dumps(EXAMPLE_STAGE1, ensure_ascii=False)}
    """

In [17]:
values_extraction_prompt = f"""
        Please extract all information regarding only the fields in EXTRACTED_FIELDS 
        from text DISASTER_DESCRIPTION (refering the information in FIELD_SCHEMA and using only 
        the categorical options in CATEGORICAL_VALUES). 
        Finally return a single JSON object that conforms precisely to OUTPUT_VALUES.
        Below there is a reference EXAMPLE_STAGE2 for one disaster's description and its output values.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        CATEGORICAL_VALUES:
        {json.dumps(CATEGORICAL_VALUES, ensure_ascii=False)}

        OUTPUT_VALUES:
        {json.dumps(OUTPUT_VALUES, ensure_ascii=False)}
            
        EXAMPLE_STAGE2:
        {json.dumps(EXAMPLE_STAGE2, ensure_ascii=False)}
    """

# Implement

In [19]:
def call_llm(prompt_text: str, description_text: str, extracted_fields_text = None) -> dict:
    
    if extracted_fields_text is None:
        user_content = description_text or ""
    else:
        user_content = json.dumps({
            "DISASTER_DESCRIPTION": description_text,
            "EXTRACTED_FIELDS": extracted_fields_text
        }, ensure_ascii=False)
        
    try:
        resp = openai.chat.completions.create(
            model           = MODEL_NAME,
            messages        = [
                {"role": "system", "content": prompt_text},
                {"role": "user", "content": user_content}
            ],
            temperature     = TEMPERATURE,
            response_format = {"type": "json_object"}
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print("LLM Error:", e)
        return None

In [20]:
fields_results = []
values_results = []
for desc in tqdm(df["description"], desc="Two-stage extracting"):
    # Stage1: fields
    p1_resp       = call_llm(fields_extraction_prompt, desc)

    fields_results.append(json.dumps(p1_resp, ensure_ascii=False)) 
    
    # Stage2: values
    p2_resp       = call_llm(values_extraction_prompt, desc, p1_resp)
    values_results.append(p2_resp)
    
    time.sleep(0.5)

Two-stage extracting:  21%|███▎            | 327/1588 [23:51<1:36:14,  4.58s/it]

LLM Error: Out of range float values are not JSON compliant
LLM Error: Out of range float values are not JSON compliant


Two-stage extracting:  23%|███▌            | 359/1588 [26:04<1:31:04,  4.45s/it]

LLM Error: Out of range float values are not JSON compliant
LLM Error: Out of range float values are not JSON compliant


Two-stage extracting:  45%|████████▏         | 718/1588 [48:52<57:18,  3.95s/it]

LLM Error: Expecting property name enclosed in double quotes: line 20 column 16265 (char 16877)


Two-stage extracting: 100%|███████████████| 1588/1588 [1:52:49<00:00,  4.26s/it]


In [21]:
df["two_stage_fields_1shot"] = fields_results
df["two_stage_values_1shot"] = values_results  

df.to_csv(out_file, index=False)
print(f"Saved to {out_file}")

Saved to extracted_twostage_1shot.csv
